In [ ]:
# Contributors: [Group 5 — fill in names]
# Course: Recommendation Engines, IE University 2025-26
# Notebook 04: Context-Aware Recommender

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from utils import *

In [ ]:
import gc
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from surprise import Dataset, Reader, SVD, accuracy

np.random.seed(42)

## 1. Data Loading & Train/Test Split

In [ ]:
df, df_meta = load_data()
print(f'Reviews: {len(df):,}')

In [ ]:
train_df, test_df, split_info = temporal_train_test_split(df)
train_items = split_info['train_items']

In [ ]:
# Add temporal features before building lookups (context model needs them)
train_df = add_temporal_features(train_df)
test_df  = add_temporal_features(test_df)

In [ ]:
lookups = build_lookup_structures(train_df, test_df)
GLOBAL_MEAN      = lookups['global_mean']
train_user_items = lookups['train_user_items']

sample_users = sample_eval_users(lookups, n=1000)

## 2. Context Feature Engineering

Context variables:
- **month** (1–12) — seasonal purchasing patterns
- **dayofweek** (0=Mon … 6=Sun) — weekday vs. weekend browsing
- **time_period** (early / mid / late) — evolving preferences over the dataset timeline

Context key = `(month, dayofweek, time_period)` — 252 unique combinations in training data.

In [ ]:
train_min_ts = int(train_df['timestamp'].min())
train_max_ts = int(train_df['timestamp'].max())
ts_range     = train_max_ts - train_min_ts

def get_time_period(ts):
    frac = (ts - train_min_ts) / ts_range
    if frac < 0.33:
        return 'early'
    elif frac < 0.66:
        return 'mid'
    return 'late'

train_df['time_period'] = train_df['timestamp'].apply(get_time_period)
test_df['time_period']  = test_df['timestamp'].apply(
    lambda ts: get_time_period(min(ts, train_max_ts))
)

train_df['context_key'] = list(zip(train_df['month'], train_df['dayofweek'], train_df['time_period']))
test_df['context_key']  = list(zip(test_df['month'],  test_df['dayofweek'],  test_df['time_period']))

print(f'Unique context keys (train): {train_df["context_key"].nunique()}')

In [ ]:
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print('Ratings by day of week:')
for dow, row in train_df.groupby('dayofweek')['rating'].agg(['mean', 'count']).iterrows():
    print(f'  {dow_names[dow]}: avg={row["mean"]:.3f}  n={int(row["count"]):,}')
print('\nRatings by time period:')
for tp, row in train_df.groupby('time_period')['rating'].agg(['mean', 'count']).iterrows():
    print(f'  {tp:5s}: avg={row["mean"]:.3f}  n={int(row["count"]):,}')

## 3. Context Bias Estimation + Residual SVD

**Approach** (pre/post-filtering hybrid):
1. Compute `context_bias(ctx) = mean_rating(ctx) − global_mean` from training data.
2. Remove bias: `rating_debiased = rating − context_bias`.
3. Train SVD on de-biased ratings.
4. At inference: `prediction = SVD(debiased) + context_bias(ctx)`.

In [ ]:
context_bias = (
    train_df.groupby('context_key')['rating'].mean() - GLOBAL_MEAN
).to_dict()

print(f'Context biases computed: {len(context_bias)} unique contexts')
print(f'Bias range: [{min(context_bias.values()):.4f}, {max(context_bias.values()):.4f}]')

train_df['context_bias']    = train_df['context_key'].map(context_bias).fillna(0.0)
train_df['rating_debiased'] = (train_df['rating'] - train_df['context_bias']).clip(1.0, 5.0)

In [ ]:
reader_ctx = Reader(rating_scale=(1.0, 5.0))
train_ctx_surprise = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating_debiased']].rename(columns={'rating_debiased': 'rating'}),
    reader_ctx
).build_full_trainset()

gc.collect()
print('Training SVD on context-debiased ratings (n_factors=30, n_epochs=20)...')
t0 = time.time()
algo_ctx = SVD(n_factors=30, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo_ctx.fit(train_ctx_surprise)
print(f'  Done in {time.time()-t0:.1f}s')
gc.collect()

In [ ]:
def predict_context_aware(user_id, item_id, context_key):
    svd_pred = algo_ctx.predict(user_id, item_id).est
    bias     = context_bias.get(context_key, 0.0)
    return float(np.clip(svd_pred + bias, 1.0, 5.0))


def get_context_top_k(user_id, context_key, k=K):
    seen       = train_user_items.get(user_id, set())
    candidates = [it for it in train_items if it not in seen]
    if len(candidates) > 2000:
        candidates = list(np.random.choice(candidates, 2000, replace=False))
    scores = [(it, predict_context_aware(user_id, it, context_key)) for it in candidates]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [it for it, _ in scores[:k]]

## 4. Evaluation

In [ ]:
print('Rating prediction on full test set...')
actual_ctx    = test_df['rating'].tolist()
predicted_ctx = [
    predict_context_aware(row.user_id, row.item_id, row.context_key)
    for row in test_df.itertuples()
]
rmse_ctx = rmse(actual_ctx, predicted_ctx)
mae_ctx  = mae(actual_ctx, predicted_ctx)
print(f'  Context-Aware — RMSE: {rmse_ctx:.4f}, MAE: {mae_ctx:.4f}')

In [ ]:
# Use each user's most recent test context for ranking recommendations
user_latest_ctx = (
    test_df.sort_values('timestamp')
    .groupby('user_id')['context_key'].last()
    .to_dict()
)
default_ctx = (6, 3, 'late')

print('Computing ranking metrics (1000 users)...')
fn_ctx = lambda uid: get_context_top_k(uid, user_latest_ctx.get(uid, default_ctx))
all_recs_ctx, ranking_ctx = evaluate_ranking(fn_ctx, sample_users, lookups)

recs_ctx_dict = {u: r for u, r in zip(sample_users, all_recs_ctx)}
results_ctx   = {'RMSE': rmse_ctx, 'MAE': mae_ctx, 'Diversity': None, **ranking_ctx}

print_metrics('Context-Aware', rmse_ctx, mae_ctx, ranking_ctx)

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Context-Aware Recommender Analysis', fontsize=13, fontweight='bold')

# Rating bias by month
monthly_bias  = train_df.groupby('month')['rating'].mean() - GLOBAL_MEAN
month_labels  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
colors_month  = ['coral' if b > 0 else 'steelblue' for b in monthly_bias.values]
axes[0,0].bar(range(1, 13), monthly_bias.values, color=colors_month, edgecolor='white')
axes[0,0].set_xticks(range(1, 13))
axes[0,0].set_xticklabels(month_labels, fontsize=8)
axes[0,0].set_title('Rating Bias by Month')
axes[0,0].set_ylabel('Bias vs global mean')
axes[0,0].axhline(0, color='gray', linestyle='--', linewidth=0.5)

# Rating bias by day of week
dow_bias   = train_df.groupby('dayofweek')['rating'].mean() - GLOBAL_MEAN
colors_dow = ['coral' if b > 0 else 'steelblue' for b in dow_bias.values]
axes[0,1].bar(range(7), dow_bias.values, color=colors_dow, edgecolor='white')
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels(dow_names)
axes[0,1].set_title('Rating Bias by Day of Week')
axes[0,1].set_ylabel('Bias vs global mean')
axes[0,1].axhline(0, color='gray', linestyle='--', linewidth=0.5)

# Actual vs predicted
axes[1,0].hist(actual_ctx[:50000],    bins=20, alpha=0.6, label='Actual',          color='steelblue', edgecolor='white')
axes[1,0].hist(predicted_ctx[:50000], bins=20, alpha=0.6, label='Predicted (Ctx)', color='coral',     edgecolor='white')
axes[1,0].set_title('Context-Aware: Actual vs Predicted')
axes[1,0].set_xlabel('Rating')
axes[1,0].set_ylabel('Count')
axes[1,0].legend()

# Placeholder RMSE comparison (full comparison in notebook 05)
model_names = ['Context-Aware']
rmse_vals   = [rmse_ctx]
axes[1,1].barh(model_names, rmse_vals, color=['coral'], edgecolor='white')
axes[1,1].set_title('RMSE — Context-Aware (full comparison in notebook 05)')
axes[1,1].set_xlabel('RMSE')

plt.tight_layout()
plt.show()

## 6. Save Results

In [ ]:
import os, pickle

output = {
    'results_ctx':   results_ctx,
    'context_bias':  context_bias,
    'recs_ctx':      recs_ctx_dict,   # {user_id: [item_ids]}
    'sample_users':  sample_users,
}

os.makedirs('../results', exist_ok=True)
with open('../results/04_context_aware.pkl', 'wb') as f:
    pickle.dump(output, f)

print('Saved: ../results/04_context_aware.pkl')